In [1]:
from pathlib import Path
import json
import shutil
from PIL import Image

# Main web app folder
WEB_ROOT = Path(r"D:\Bank_web1")

# Existing manifest
MANIFEST_PATH = WEB_ROOT / "data" / "sample-images.json"

# Destination folders used by the app
EXAMPLES_DIR = WEB_ROOT / "img" / "examples"
THUMBS_DIR = EXAMPLES_DIR / "thumbs"

EXAMPLES_DIR.mkdir(parents=True, exist_ok=True)
THUMBS_DIR.mkdir(parents=True, exist_ok=True)

# Keep original_name in the public manifest?
# I recommend False because colony IDs may be sensitive.
KEEP_ORIGINAL_NAME = False

# Thumbnail settings
THUMB_MAX_SIZE = (520, 360)
JPEG_QUALITY = 88

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

new_manifest = []

for i, item in enumerate(manifest, start=1):
    example_id = item.get("id", f"example_{i:02d}")
    source_path = Path(item["source_path"])

    if not source_path.exists():
        raise FileNotFoundError(f"Source image not found: {source_path}")

    # Keep clean standard names for GitHub/web use
    image_name = f"{example_id}.jpg"
    thumb_name = f"{example_id}_thumb.jpg"

    image_dst = EXAMPLES_DIR / image_name
    thumb_dst = THUMBS_DIR / thumb_name

    # Open and convert to RGB to avoid format/mode issues on GitHub Pages
    with Image.open(source_path) as img:
        img = img.convert("RGB")

        # Save web image
        img.save(image_dst, "JPEG", quality=JPEG_QUALITY, optimize=True)

        # Save thumbnail
        thumb = img.copy()
        thumb.thumbnail(THUMB_MAX_SIZE)
        thumb.save(thumb_dst, "JPEG", quality=JPEG_QUALITY, optimize=True)

    clean_item = {
        "id": example_id,
        "file": f"img/examples/{image_name}",
        "thumb": f"img/examples/thumbs/{thumb_name}",
    }

    if KEEP_ORIGINAL_NAME and "original_name" in item:
        clean_item["original_name"] = item["original_name"]

    new_manifest.append(clean_item)

# Rewrite safe public manifest
with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(new_manifest, f, indent=2, ensure_ascii=False)

print(f"Copied {len(new_manifest)} example images to: {EXAMPLES_DIR}")
print(f"Created thumbnails in: {THUMBS_DIR}")
print(f"Updated manifest: {MANIFEST_PATH}")

print("\nUpdated manifest preview:")
print(json.dumps(new_manifest[:3], indent=2, ensure_ascii=False))

Copied 10 example images to: D:\Bank_web1\img\examples
Created thumbnails in: D:\Bank_web1\img\examples\thumbs
Updated manifest: D:\Bank_web1\data\sample-images.json

Updated manifest preview:
[
  {
    "id": "example_01",
    "file": "img/examples/example_01.jpg",
    "thumb": "img/examples/thumbs/example_01_thumb.jpg"
  },
  {
    "id": "example_02",
    "file": "img/examples/example_02.jpg",
    "thumb": "img/examples/thumbs/example_02_thumb.jpg"
  },
  {
    "id": "example_03",
    "file": "img/examples/example_03.jpg",
    "thumb": "img/examples/thumbs/example_03_thumb.jpg"
  }
]


In [2]:
from pathlib import Path
import json

WEB_ROOT = Path(r"D:\Bank_web1")
MANIFEST_PATH = WEB_ROOT / "data" / "sample-images.json"

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

missing = []

for item in manifest:
    image_path = WEB_ROOT / item["file"]
    thumb_path = WEB_ROOT / item["thumb"]

    if not image_path.exists():
        missing.append(str(image_path))

    if not thumb_path.exists():
        missing.append(str(thumb_path))

print("Examples in manifest:", len(manifest))
print("Missing files:", len(missing))

if missing:
    for path in missing:
        print("Missing:", path)
else:
    print("All example images and thumbnails exist.")

Examples in manifest: 10
Missing files: 0
All example images and thumbnails exist.
